# BELEN on Colab

TwoQuarks Research · Luis Jaime Ledesma Pérez

Before anything: **Runtime → Change runtime type → A100 GPU**.

Each cell is a rung of a ladder. **If a cell ends in red, do not run the next one.** Copy the error and fix it there. The runner also refuses on its own: a stage does not start if the one before it is not green.

The code always comes from GitHub, never from Drive. Drive only keeps data, weights and logs, so a disconnection does not cost you the run.

## 1. Bring the code

In [ ]:
REPO_URL = "https://github.com/Jaime2pb3/BELEN_LM.git"
BRANCH = "main"
!rm -rf /content/BELEN && git clone --depth 1 --branch $BRANCH $REPO_URL /content/BELEN
%cd /content/BELEN
!git log -1 --format="commit %h · %cd"
!pip -q install sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Work folder and model arm

`WORK` holds weights and results, one folder per arm. The data is downloaded once and shared by every arm. Mount Drive if you want all of it to survive a disconnection.

The arm decides which parts of BELEN are on. Change one thing at a time.

| Arm | Phase 3 | Geometric dendrite |
|---|---|---|
| `base` | axon views | off |
| `geometry` | axon views | on |
| `ach` | cholinergic gate | off |
| `ach_geometry` | cholinergic gate | on |

In [ ]:
USE_DRIVE = False
ARM = "base"
ITERATIONS = 6500

WORK = "/content/belen_work"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/belen_work"

ARMS = {
    "base": {"AXON_MODE": "views", "GEOMETRIC_DENDRITE": "0"},
    "geometry": {"AXON_MODE": "views", "GEOMETRIC_DENDRITE": "1"},
    "ach": {"AXON_MODE": "ach", "GEOMETRIC_DENDRITE": "0"},
    "ach_geometry": {"AXON_MODE": "ach", "GEOMETRIC_DENDRITE": "1"},
}
import os
os.environ.update(ARMS[ARM])
ENV_FLAGS = " ".join(f"--env {k}={v}" for k, v in ARMS[ARM].items())
DATA_HOME = f"{WORK}/data"
WORK = f"{WORK}/{ARM}"
os.makedirs(DATA_HOME, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f"{WORK}/data"):
    os.symlink(DATA_HOME, f"{WORK}/data")
print("work :", WORK)
print("data :", DATA_HOME, "(shared by every arm)")
print("arm  :", ENV_FLAGS)

## 3. The model checks itself (CPU, a few minutes)

The selftest, the regression against the audited base, and the six combinations of Phase 3 and geometry.

In [ ]:
!python train_belen.py --selftest
!python tests/test_regression.py
!python tests/test_ach_and_geometry.py
!python train_belen.py --params

## 4. Cheap rungs: environment, selftests, data, wiring

The 104M model at 6500 iterations × 65536 tokens reads about 426M tokens, that is 5 shards (about 1 GB). There are 195 available.

In [ ]:
!python eval_pii/ladder.py env --work $WORK
!python eval_pii/ladder.py selftest --work $WORK
!python eval_pii/ladder.py data --work $WORK --train-shards 5
!python eval_pii/ladder.py wiring --work $WORK $ENV_FLAGS

## 5. Preflight, the first rung that touches the GPU

The real model at real size for 2 iterations, with the real batch and sequence. It catches out of memory, data and checkpoint problems, and builds a smoke card with the real model.

**If it breaks here, it will break worse in the full run.**

In [ ]:
!python eval_pii/ladder.py preflight --work $WORK $ENV_FLAGS

## 6. Training

Partial validation every 500 steps, a resumable checkpoint every 250 (if Colab disconnects, run this cell again and it continues) and, at the end, the ablation report: full model, frozen memory, one averaged slot, memory at zero, and no axon or gate.

Test time training only acts during evaluation and is off here, so the saved weights are the trained ones, not the adapted ones.

In [ ]:
!python eval_pii/ladder.py train --work $WORK --iterations $ITERATIONS $ENV_FLAGS

## 7. Preregistration

Criteria, predictions and the hashes of all the code, written **before** measuring. It is written once.

In [ ]:
!python eval_pii/ladder.py prereg --work $WORK --langs en
!cat $WORK/ficha/PREREG.md

**Your turn:** open `PREREG.md` (files panel, double click), write your predictions in `J1` and `J2`, save. Then measure.

## 8. The card

In [ ]:
!python eval_pii/ladder.py ficha --work $WORK

## 8b. Text fingerprint (optional, exploratory, no verdict)

What the model knows about a text reading only 4, 8, 16 or 32 tokens. Language and carrier are trivial and are there so that triviality stays visible. The one that matters is word order on real FineWeb fragments.

In [ ]:
!python eval_pii/ladder.py huella --work $WORK

## 9. Results

In [ ]:
!python eval_pii/ladder.py pack --work $WORK
!python eval_pii/ladder.py status --work $WORK
!ls $WORK

## 10. Paired comparison (optional, long)

The same seed, data and schedule for each arm, one factor at a time. It reuses the data downloaded in step 4. Each arm costs a full training run.

In [ ]:
DATA = f"{WORK}/data"
!DATA_PATH=$DATA/datasets/fineweb10B_sp1024/ TOKENIZER_PATH=$DATA/tokenizers/fineweb_1024_bpe.model ITERATIONS=$ITERATIONS VAL_LOSS_EVERY=500 CKPT_DIR=$WORK/ab_ckpt python tools/ab_runs.py --arms base geometry ach ach_geometry --seeds 42